<a href="https://colab.research.google.com/github/MightyNine/The-Mighty-Nein/blob/main/The_GAN_Model_(HAYDEN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Checking columns and values in the dataset

In [1]:
import pandas as pd
df = pd.read_csv("train.csv")
print("Columns:", df.columns.tolist())
print("Value counts:")
print(df['label'].value_counts())

Columns: ['flow_duration', 'Header_Length', 'Protocol Type', 'Duration', 'Rate', 'Srate', 'Drate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'urg_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Magnitue', 'Radius', 'Covariance', 'Variance', 'Weight', 'label']
Value counts:
label
DDoS-ICMP_Flood            848088
DDoS-UDP_Flood             637558
DDoS-TCP_Flood             528499
DDoS-PSHACK_Flood          481254
DDoS-SYN_Flood             478653
DDoS-RSTFINFlood           475441
DDoS-SynonymousIP_Flood    422083
DoS-UDP_Flood              390422
DoS-TCP_Flood              314174
DoS-SYN_Flood              237573
BenignTraffic              129538
Mirai-greeth_flood         116133
Mirai-udpplain             104814

Loading the dataset and conducting dataset training

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset


# 1. Dataset Class with Hyperparameters to Limit Dataset Size
class AnomalyDataset(Dataset):

    def __init__(
        self,
        csv_file,
        label_col="label ",
        normal_label="Benign",
        target_shape=(16, 16),
        max_samples=None,
        subset_fraction=None,
    ):
        self.df = pd.read_csv(csv_file)

        # Filter strictly for normal data
        if label_col in self.df.columns:
            self.df = self.df[self.df[label_col] == normal_label].reset_index(
                drop=True
            )
            self.df = self.df.drop(columns=[label_col])

        # Hyperparameter 1: Take a random fraction of the dataset (e.g., 0.10 = 10%)
        if subset_fraction is not None and 0.0 < subset_fraction < 1.0:
            self.df = self.df.sample(
                frac=subset_fraction, random_state=42
            ).reset_index(drop=True)

        # Hyperparameter 2: Cap total samples to a fixed maximum number (e.g., 50000)
        if max_samples is not None and len(self.df) > max_samples:
            self.df = self.df.iloc[:max_samples].reset_index(drop=True)

        # Keep ONLY numeric features (removes string attack names, IPs, etc.)
        self.df = self.df.select_dtypes(include=[np.number])

        # Clean invalid values (infinity / NaN)
        self.df = self.df.replace([np.inf, -np.inf], np.nan)
        self.df = self.df.fillna(self.df.median())

        # Scale features to [-1, 1] range
        data_min = self.df.min()
        data_max = self.df.max()
        range_diff = (data_max - data_min).replace(0, 1)

        scaled_features = 2 * (self.df - data_min) / range_diff - 1
        scaled_numpy = scaled_features.to_numpy(dtype=np.float32)

        # Pad features with zeros to fit the target 2D image shape (16 x 16 = 256)
        total_pixels = target_shape[0] * target_shape[1]
        num_samples, num_features = scaled_numpy.shape

        padded_features = np.zeros(
            (num_samples, total_pixels), dtype=np.float32
        )
        padded_features[:, :num_features] = scaled_numpy

        # Reshape to (N, C, H, W) -> (N, 1, 16, 16)
        reshaped_data = padded_features.reshape(
            -1, 1, target_shape[0], target_shape[1]
        )
        self.data = torch.tensor(reshaped_data, dtype=torch.float32)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


# 2. Generator Network (DCGAN Architecture)
class Generator(nn.Module):

    def __init__(self, z_dim=100, img_channels=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 256, kernel_size=4, stride=1, padding=0),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(
                128, img_channels, kernel_size=4, stride=2, padding=1
            ),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)


# 3. Discriminator Network (DCGAN Architecture)
class Discriminator(nn.Module):

    def __init__(self, img_channels=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(img_channels, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, kernel_size=4, stride=1, padding=0),
            nn.Sigmoid(),
        )

    def forward(self, img):
        return self.net(img)


# ==========================================
# 4. Dataset Hyperparameters for Speedup
# ==========================================
DATASET_CSV = "train.csv"

# Hyperparameters to cut down processing time:
SUBSET_FRACTION = 0.10  # Loads 10% of the normal dataset
MAX_SAMPLES = (
    50000  # Caps total training samples (set to None if using fraction only)
)

dataset = AnomalyDataset(
    csv_file=DATASET_CSV,
    label_col="label",
    normal_label="BenignTraffic",
    target_shape=(16, 16),
    max_samples=MAX_SAMPLES,
    subset_fraction=SUBSET_FRACTION,
)

BATCH_SIZE = 128  # Larger batch size speeds up GPU training
dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2
)

print(
    f"Successfully loaded {len(dataset)} normal training samples from {DATASET_CSV}."
)

# 5. Device & Model Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Z_DIM = 100

generator = Generator(z_dim=Z_DIM, img_channels=1).to(device)
discriminator = Discriminator(img_channels=1).to(device)

criterion = nn.BCELoss()
optimizer_G = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

# 6. Training Execution Loop
EPOCHS = 5

for epoch in range(EPOCHS):
    for batch_idx, real_images in enumerate(dataloader):
        batch_size = real_images.size(0)
        real_images = real_images.to(device)

        real_labels = torch.ones(batch_size, 1, 1, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1, 1, 1).to(device)

        # ----------------------
        #  Train Discriminator
        # ----------------------
        optimizer_D.zero_grad()

        output_real = discriminator(real_images)
        loss_D_real = criterion(output_real, real_labels)

        z = torch.randn(batch_size, Z_DIM, 1, 1).to(device)
        fake_images = generator(z)
        output_fake = discriminator(fake_images.detach())
        loss_D_fake = criterion(output_fake, fake_labels)

        loss_D = loss_D_real + loss_D_fake
        loss_D.backward()
        optimizer_D.step()

        # ----------------------
        #  Train Generator
        # ----------------------
        optimizer_G.zero_grad()

        output_fake_for_G = discriminator(fake_images)
        loss_G = criterion(output_fake_for_G, real_labels)

        loss_G.backward()
        optimizer_G.step()

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | Loss D: {loss_D.item():.4f} | Loss G: {loss_G.item():.4f}"
    )

Successfully loaded 12954 normal training samples from train.csv.


In [ ]:
# ================================
# MARTIN - TEST DATASET EVALUATION
# ================================

import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Load test.csv
test_df = pd.read_csv("test.csv")

print("Test dataset loaded.")
print("Rows:", len(test_df))

# 0 = Benign, 1 = Anomaly/Attack
true_labels = (test_df["label"] != "BenignTraffic").astype(int).to_numpy()

# Remove label column
test_features = test_df.drop(columns=["label"])

# Keep numeric columns only
test_features = test_features.select_dtypes(include=[np.number])

# Clean bad values
test_features = test_features.replace([np.inf, -np.inf], np.nan)
test_features = test_features.fillna(test_features.median())

# Scale values to [-1, 1]
data_min = test_features.min()
data_max = test_features.max()
range_diff = (data_max - data_min).replace(0, 1)

test_scaled = 2 * (test_features - data_min) / range_diff - 1

# Convert data into 16 x 16 format
test_numpy = test_scaled.to_numpy(dtype=np.float32)

total_pixels = 16 * 16

padded_test = np.zeros(
    (len(test_numpy), total_pixels),
    dtype=np.float32
)

padded_test[:, :test_numpy.shape[1]] = test_numpy

test_tensor = torch.tensor(
    padded_test.reshape(-1, 1, 16, 16),
    dtype=torch.float32
)

test_loader = DataLoader(
    test_tensor,
    batch_size=128,
    shuffle=False
)

# Run test data through trained discriminator
discriminator.eval()

scores = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)

        output = discriminator(batch)

        scores.extend(
            output.view(-1).cpu().numpy()
        )

scores = np.array(scores)

# Anything below 0.5 = anomaly
threshold = 0.5

predicted_labels = (scores < threshold).astype(int)

# Print results
print("\n--- GAN TEST RESULTS ---")

print("Accuracy:", accuracy_score(true_labels, predicted_labels))
print("Precision:", precision_score(true_labels, predicted_labels))
print("Recall:", recall_score(true_labels, predicted_labels))
print("F1 Score:", f1_score(true_labels, predicted_labels))

print("\nClassification Report:")
print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=["Benign", "Anomaly"]
    )
)